Step 1: Install, bootstrap instructions

In [ ]:
!pip install --quiet anthropic pydantic
from google.colab import userdata, drive # type: ignore
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")

from typing import Any
result: dict[str, Any] = {}

# Sanity import
from astra_swarm.agent_loop import run_with_tools
from astra_swarm.tools import ALL_TOOL_SCHEMAS
from astra_swarm.cassette import cassette
print("ready — tools:", [t["name"] for t in ALL_TOOL_SCHEMAS])

Step 2: One round-trip example

In [ ]:
with cassette("day03_Explain_Technique", mode="auto"):
    result = run_with_tools(
        "In two sentences, explain what MITRE ATT&CK technique T1078 is and how a SOC "
        "would detect it. Use the lookup tool for authoritative context.",
        verbose=True,
)

print()
print("=== FINAL ANSWER ===")
print(result["final_text"])
print()
print(f"rounds: {result['rounds']}, stop_reason: {result['stop_reason']}")

Step 3: Inspect the raw messages

In [ ]:
import json

def _serialize_block(b):
    """Anthropic content blocks aren't JSON-native; extract just the fields we care about."""
    if isinstance(b, dict):
        return b  # tool_result blocks are already dicts
    d = {"type": b.type}
    if b.type == "text":
        d["text"] = b.text
    elif b.type == "tool_use":
        d["id"] = b.id
        d["name"] = b.name
        d["input"] = b.input
    return d

for i, msg in enumerate(result["messages"]):
    role = msg["role"]
    content = msg["content"]
    print(f"--- turn {i}: role={role} ---")
    if isinstance(content, str):
        print(f"  (string) {content[:200]}")
    else:
        for b in content:
            print(f"  {_serialize_block(b)}")
    print()

Step 4: Model can choose not to use the tool

In [ ]:
with cassette("day03_no_tool", mode="auto"):
    result = run_with_tools(
        "In one sentence, what year was the Anthropic company founded? "
        "Do not use any tools unless strictly necessary.",
        verbose=True,
    )
print()
print("final:", result["final_text"])
print(f"rounds: {result['rounds']}, stop_reason: {result['stop_reason']}")

Step 5: Force multi-tool exchange and multi-round behavior

In [ ]:
with cassette("day03_Compare_T1078_T1110", mode="auto"):
    result = run_with_tools(
        "Compare MITRE ATT&CK T1078 and T1110 in two paragraphs — what each covers, "
        "and how a SOC analyst tells them apart in practice. Use the lookup tool for both.",
        verbose=True,
        max_rounds=6,
    )

print()
print(result["final_text"])
print(f"\nrounds: {result['rounds']}, stop_reason: {result['stop_reason']}")

Step 6: Error scenatio

In [ ]:
with cassette("day03_Error_Scenario", mode="auto"):
    result = run_with_tools(
        "Look up MITRE ATT&CK technique T9999 and summarize it.",
        verbose=True,
    )
print(result["final_text"])

Step 7: Unmount & cleanup

In [ ]:
drive.flush_and_unmount()